# NMN - polycomb

In [ ]:
# ============================================================================
# MASTER PIPELINE — AGING (GSE190102 K27me3) vs NMN-RESCUE (GSE85718)
# Self-contained: builds BOTH gene programs from raw data, then compares them.
# Produces one very detailed MASTER_SUMMARY.txt at the end.
# Run as a single Colab cell.
# ============================================================================

# ---------- 0. Install dependencies ----------
import subprocess, sys
subprocess.run("pip -q install GEOparse gseapy statsmodels scikit-learn "
               "seaborn matplotlib-venn pyBigWig pyranges gtfparse tqdm requests",
               shell=True, check=False)
subprocess.run("apt-get -qq install -y wget > /dev/null", shell=True, check=False)

import os, re, glob, shutil, textwrap, datetime, warnings, traceback
import numpy as np, pandas as pd
from scipy import stats
from scipy.stats import hypergeom
from sklearn.preprocessing import quantile_transform
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set(style="whitegrid"); plt.rcParams["figure.dpi"] = 130

try:
    import gseapy as gp; HAVE_GSEAPY = True
except Exception as e:
    print("gseapy unavailable:", e); HAVE_GSEAPY = False

# ---------- Master workspace + logging ----------
ROOT = "/content/AGING_vs_NMN_MASTER"
for d in [ROOT, f"{ROOT}/plots", f"{ROOT}/tables"]:
    os.makedirs(d, exist_ok=True)

RUN_LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); RUN_LOG.append(s)

# Containers for everything we will report
RES = {"nmn": {}, "aging": {}, "compare": {}}

UNIVERSE_N = 20000          # ~mouse protein-coding genes
ADJP_CUT   = 0.05

# Curated mitochondrial/OXPHOS core (upper-cased mouse symbols)
FALLBACK_MITO = {
    "NDUFA1","NDUFA2","NDUFA9","NDUFB8","NDUFS1","NDUFS3","NDUFV1","SDHA","SDHB",
    "UQCRC1","UQCRC2","CYC1","COX4I1","COX5A","COX5B","COX6A1","COX7A1","COX8B",
    "ATP5A1","ATP5B","ATP5C1","ATP5O","ATP5J","CS","IDH3A","SUCLA2","FH1","MDH2",
    "ACO2","OGDH","PDHA1","PDHB","DLAT","CPT1B","CPT2","ACADM","ACADL","ACADVL",
    "HADHA","HADHB","ECI1","SLC25A4","SLC25A20","TFAM","NRF1","PPARGC1A","ESRRA",
    "MFN1","MFN2","OPA1","DNM1L","PINK1","PRKN","SOD2","TFB1M","TFB2M","POLG",
    "MRPL12","MRPS12","TIMM23","TOMM20","VDAC1","VDAC2","CYCS","UCP2","UCP3",
}

ENRICHR_LIBS = ["GO_Biological_Process_2023", "KEGG_2019_Mouse",
                "Reactome_2022", "MSigDB_Hallmark_2020",
                "MGI_Mammalian_Phenotype_Level_4_2021"]

def run_enrichr(gene_list, tag):
    """Robust Enrichr wrapper -> concatenated results df (or empty)."""
    if not HAVE_GSEAPY:
        log(f"  [{tag}] Enrichr skipped (no gseapy)."); return pd.DataFrame()
    gl = list(dict.fromkeys([g for g in gene_list if g]))
    if len(gl) < 5:
        log(f"  [{tag}] Enrichr skipped (<5 genes)."); return pd.DataFrame()
    if len(gl) > 2000: gl = gl[:2000]
    frames = []
    for lib in ENRICHR_LIBS:
        ok = False
        for attempt in range(3):
            try:
                e = gp.enrichr(gene_list=gl, gene_sets=lib,
                               organism="mouse", outdir=None)
                cand = e.results.copy()
                if "Adjusted P-value" in cand.columns and len(cand):
                    cand["Library"] = lib; frames.append(cand)
                    nsig = int((cand["Adjusted P-value"] < ADJP_CUT).sum())
                    log(f"  [{tag}] {lib:42s} terms={len(cand):5d}  sig={nsig}")
                    ok = True; break
                raise ValueError("no Adjusted P-value column")
            except Exception as ex:
                import time; time.sleep(3*(attempt+1))
        if not ok:
            log(f"  [{tag}] {lib:42s} FAILED after retries.")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ============================================================================
# PART A —  GSE85718  : NMN-RESCUE TRANSCRIPTOMIC PROGRAM (from raw)
# ============================================================================
log("\n" + "#"*78)
log("# PART A — GSE85718 NMN aging microarray  (building rescue gene program)")
log("#"*78)

NMN_OK = False
try:
    import GEOparse
    NWORK = f"{ROOT}/GSE85718"; os.makedirs(NWORK + "/raw", exist_ok=True)
    log(">>> Downloading & parsing GSE85718 (SOFT family file)...")
    gse = GEOparse.get_GEO(geo="GSE85718", destdir=NWORK + "/raw", silent=True)

    expr_probe = gse.pivot_samples("VALUE").apply(pd.to_numeric, errors="coerce")
    log(">>> Raw probe matrix:", expr_probe.shape)

    gpl = list(gse.gpls.values())[0]
    ann = gpl.table.copy()
    id_col  = "ID" if "ID" in ann.columns else ann.columns[0]
    sym_col = next((c for c in ["Symbol","ILMN_Gene","Gene_symbol","GENE_SYMBOL"]
                    if c in ann.columns), None)
    assert sym_col, f"No symbol column in GPL: {list(ann.columns)}"
    probe2sym = ann.set_index(id_col)[sym_col].astype(str).to_dict()

    # --- metadata from titles: "Skeletal muscle, control, 6M, rep1" ---
    rows = []
    for gsm_name, gsm in gse.gsms.items():
        title = gsm.metadata["title"][0]; t = title.lower()
        tissue = ("Skeletal_muscle" if "muscle" in t else
                  "Liver" if "liver" in t else "WAT" if "wat" in t else None)
        treatment = "NMN" if "nmn" in t else ("Control" if "control" in t else None)
        age = ("6M" if re.search(r"\b6m\b", t) else
               "12M" if re.search(r"\b12m\b", t) else None)
        rep = re.search(r"rep\s*(\d+)", t)
        rows.append({"sample_id": gsm_name, "tissue": tissue,
                     "treatment": treatment, "age": age,
                     "rep": int(rep.group(1)) if rep else np.nan})
    meta = pd.DataFrame(rows).set_index("sample_id").loc[expr_probe.columns]
    meta["age_num"]   = meta["age"].map({"6M": 0, "12M": 1})
    meta["treat_num"] = meta["treatment"].map({"Control": 0, "NMN": 1})
    assert meta[["tissue","treatment","age"]].isna().sum().sum() == 0, \
        "Some titles failed to parse."
    RES["nmn"]["design"] = meta.groupby(["tissue","age","treatment"]).size().to_dict()
    log(">>> Sample design:", RES["nmn"]["design"])

    # --- collapse probes -> genes (highest-mean probe per symbol) ---
    sym = expr_probe.index.map(probe2sym)
    keep = pd.notna(sym) & ~pd.Series(sym, index=expr_probe.index).isin(
            ["nan","NA","","---"])
    eg = expr_probe.loc[keep].copy()
    eg["__sym__"]  = sym[keep]
    eg["__mean__"] = eg.drop(columns="__sym__").mean(axis=1)
    n_probes_in = expr_probe.shape[0]
    eg = (eg.sort_values("__mean__", ascending=False)
            .drop_duplicates("__sym__").set_index("__sym__").drop(columns="__mean__"))
    eg.index.name = "gene"
    log(f">>> Gene-level matrix: {eg.shape} (from {n_probes_in} probes)")

    # --- normalization: log2 (guarded) + per-sample quantile-normal ---
    mat = eg.copy()
    if np.nanmax(mat.values) > 100:
        mat = np.log2(mat.clip(lower=0) + 1)
        RES["nmn"]["norm_note"] = "log2(x+1) applied + quantile-normal"
    else:
        RES["nmn"]["norm_note"] = "log2 skipped (already compressed) + quantile-normal"
    norm = pd.DataFrame(
        quantile_transform(mat.values, axis=0, output_distribution="normal", copy=True),
        index=mat.index, columns=mat.columns)

    # --- vectorized per-tissue OLS:  expr ~ age * treatment ---
    def vectorized_ols(Y, X):
        n, p = X.shape
        XtX_inv = np.linalg.inv(X.T @ X)
        beta = Y @ X @ XtX_inv
        resid = Y - beta @ X.T
        dof = n - p
        sigma2 = (resid**2).sum(axis=1) / dof
        se = np.sqrt(np.outer(sigma2, np.diag(XtX_inv)))
        tvals = beta / se
        pvals = 2 * stats.t.sf(np.abs(tvals), dof)
        return beta, pvals

    def run_tissue_de(tissue):
        sub = meta[meta.tissue == tissue]
        Y = norm[sub.index].values
        a = sub["age_num"].values.astype(float)
        g = sub["treat_num"].values.astype(float)
        X = np.column_stack([np.ones_like(a), a, g, a*g])
        beta, pval = vectorized_ols(Y, X)
        de = pd.DataFrame({
            "gene": norm.index,
            "coef_age_Control": beta[:,1], "p_age_Control": pval[:,1],
            "coef_NMN_at6M":    beta[:,2], "p_NMN_at6M":    pval[:,2],
            "coef_interaction": beta[:,3], "p_interaction": pval[:,3]})
        for tag in ["age_Control","NMN_at6M","interaction"]:
            de[f"fdr_{tag}"] = multipletests(de[f"p_{tag}"], method="fdr_bh")[1]
        return de.sort_values("p_interaction").reset_index(drop=True)

    TISSUES = ["Skeletal_muscle","Liver","WAT"]
    de_results = {t: run_tissue_de(t) for t in TISSUES}
    RES["nmn"]["de_counts"] = {}
    for t,d in de_results.items():
        RES["nmn"]["de_counts"][t] = {
            "age_Control": int((d.fdr_age_Control < 0.05).sum()),
            "NMN_at6M":    int((d.fdr_NMN_at6M  < 0.05).sum()),
            "interaction": int((d.fdr_interaction < 0.05).sum())}
        log(f">>> {t:16s} sig age:{RES['nmn']['de_counts'][t]['age_Control']:5d}"
            f"  NMN@6M:{RES['nmn']['de_counts'][t]['NMN_at6M']:5d}"
            f"  interaction:{RES['nmn']['de_counts'][t]['interaction']:5d}")

    # --- NMN-rescue genes: age-sig in Control + sign-reversal interaction ---
    def get_rescue(de, fdr_age=0.05, fdr_int=0.10, min_eff=0.3, min_n=15):
        r = de[(de.fdr_age_Control < fdr_age) & (de.fdr_interaction < fdr_int) &
               (np.sign(de.coef_age_Control) != np.sign(de.coef_interaction)) &
               (de.coef_age_Control.abs() > min_eff)].copy()
        mode = "strict"
        if len(r) < min_n:
            r = de[(de.p_age_Control < 0.05) & (de.p_interaction < 0.05) &
                   (np.sign(de.coef_age_Control) != np.sign(de.coef_interaction))].copy()
            mode = "relaxed (nominal p)"
        return r.sort_values("p_interaction"), mode

    rescue, rescue_mode = {}, {}
    for t in TISSUES:
        r, mode = get_rescue(de_results[t])
        r["tissue"] = t
        r["gene"]   = r["gene"].astype(str).str.upper()
        rescue[t], rescue_mode[t] = r, mode
        log(f">>> {t:16s} NMN-rescue genes: {len(r):4d} [{mode}]")

    # --- cross-tissue robust rescue (>=2 tissues) + direction ---
    all_resc = pd.concat([rescue[t] for t in TISSUES], ignore_index=True)
    counts = all_resc.groupby("gene")["tissue"].nunique()
    robust_genes = set(counts[counts >= 2].index)
    robust_detail = (all_resc[all_resc.gene.isin(robust_genes)]
                     .groupby("gene")
                     .agg(n_tissues=("tissue","nunique"),
                          mean_coef_interaction=("coef_interaction","mean"),
                          mean_coef_age_Control=("coef_age_Control","mean"))
                     .reset_index())
    nmn_dir = dict(zip(robust_detail.gene,
                       np.sign(robust_detail.mean_coef_interaction)))
    genes_all3 = set(counts[counts == 3].index)

    nmn_union = set(all_resc.gene.unique())
    nmn_mito  = (robust_genes | nmn_union) & FALLBACK_MITO

    RES["nmn"].update({
        "rescue_mode": rescue_mode,
        "rescue_counts": {t: len(rescue[t]) for t in TISSUES},
        "robust_genes": robust_genes,
        "robust_detail": robust_detail,
        "genes_all3": genes_all3,
        "union_genes": nmn_union,
        "nmn_dir": nmn_dir,
        "nmn_mito": nmn_mito,
        "n_probes_in": n_probes_in,
        "n_genes": eg.shape[0],
    })
    robust_detail.to_csv(f"{ROOT}/tables/NMN_robust_detail.tsv", sep="\t", index=False)
    log(f">>> NMN robust (>=2 tissues): {len(robust_genes)} | all-3: {len(genes_all3)}")
    log(f">>> NMN mito overlap: {len(nmn_mito)}")
    NMN_OK = True
except Exception as e:
    log(">>> PART A FAILED:", e); log(traceback.format_exc())

# ============================================================================
# PART B —  GSE190102 : K27me3 EPIGENETIC AGING PROGRAM (ABC, from raw)
# ============================================================================
log("\n" + "#"*78)
log("# PART B — GSE190102 K27me3 ABC aging  (building epigenetic target program)")
log("#"*78)

AGING_OK = False
try:
    import pyBigWig, pyranges as pr
    from sklearn.cluster import KMeans
    from tqdm.auto import tqdm

    AWORK = f"{ROOT}/ABC_GSE190102"
    for d in ["data/bw","peaks","abc_inputs","ABC_output"]:
        os.makedirs(f"{AWORK}/{d}", exist_ok=True)
    os.chdir(AWORK)

    TAR = "data/GSE190102_RAW.tar"
    if not os.path.exists(TAR):
        log(">>> Downloading GSE190102 RAW tar (LARGE, 20-40 min)...")
        subprocess.run(
            f'wget -q -O {TAR} '
            f'"https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE190102&format=file"',
            shell=True, check=True)

    listing = [x for x in subprocess.run(f"tar -tf {TAR}", shell=True,
               capture_output=True, text=True).stdout.splitlines() if x.strip()]
    log(f">>> Files in tar: {len(listing)}")

    KNOWN_MARKS = ["K27ac","K4me3","K4me1","K36me3","K27me3","K9me3"]
    PREFERRED   = ["K27ac","K4me3","K4me1","K27me3","K9me3","K36me3"]
    ACTIVE_MARKS = {"K27ac","K4me3","K4me1"}
    def detect_mark(name):
        for mk in KNOWN_MARKS:
            if re.search(mk, name, re.I): return mk
        return None
    bw_entries = [e for e in listing if re.search(r"\.(bw|bigwig)$", e, re.I)]
    marks_present = {}
    for e in bw_entries:
        mk = detect_mark(os.path.basename(e))
        if mk: marks_present.setdefault(mk, []).append(e)
    assert marks_present, "No histone-mark bigWigs found in tar."
    MARK = next((p for p in PREFERRED if p in marks_present), None) or \
           max(marks_present, key=lambda k: len(marks_present[k]))
    MARK_IS_ACTIVE = MARK in ACTIVE_MARKS
    log(f">>> Using mark: {MARK} (active={MARK_IS_ACTIVE})")

    shutil.rmtree("data/bw", ignore_errors=True); os.makedirs("data/bw", exist_ok=True)
    with open("data/bw_list.txt","w") as fh:
        fh.write("\n".join(marks_present[MARK]) + "\n")
    subprocess.run(f"tar -xf {TAR} -C data/bw/ -T data/bw_list.txt", shell=True)
    for r,_,files in os.walk("data/bw"):
        for fn in files:
            src=os.path.join(r,fn); dst=os.path.join("data/bw",fn)
            if src!=dst and re.search(r"\.(bw|bigwig)$",fn,re.I): os.replace(src,dst)
    bw_files = sorted(glob.glob("data/bw/*.bw")+glob.glob("data/bw/*.bigwig"))
    assert bw_files, "No bigWigs extracted."
    log(f">>> {MARK} bigWigs on disk: {len(bw_files)}")

    # --- sample sheet ---
    AGE_PATTERNS = [re.compile(r"WT(\d{1,2})",re.I),
                    re.compile(r"_(\d{1,2})\s*M(?:_|\.|\b)",re.I),
                    re.compile(r"(?:age|mo|month[s]?)[_\-]?(\d{1,2})",re.I)]
    VALID_AGES = {3,12,24}
    def parse_age(name):
        for pat in AGE_PATTERNS:
            m = pat.search(name)
            if m and int(m.group(1)) in VALID_AGES: return f"{int(m.group(1))}M"
        return None
    rows=[]
    for f in bw_files:
        age = parse_age(os.path.basename(f))
        if age: rows.append({"file": f, "age": age})
    samples = pd.DataFrame(rows).sort_values(["age","file"]).reset_index(drop=True)
    assert len(samples), "No samples parsed."
    samples["rep"] = samples.groupby("age").cumcount()+1
    RES["aging"]["samples_per_age"] = samples.groupby("age").size().to_dict()
    log(">>> Samples per age:", RES["aging"]["samples_per_age"])
    assert (samples.age=="3M").any() and (samples.age=="24M").any(), \
        "Need 3M and 24M."

    CHROMS = ["chr"+c for c in list(map(str,range(1,20)))+["X"]]
    def top_bins(bw_path, bin_size=1000, top_pct=2.0):
        bw=pyBigWig.open(bw_path); out=[]
        for chrom in CHROMS:
            if chrom not in bw.chroms(): continue
            L=bw.chroms()[chrom]; nbins=L//bin_size
            if nbins==0: continue
            vals=np.nan_to_num(bw.values(chrom,0,nbins*bin_size,numpy=True))
            vals=vals.reshape(nbins,bin_size).mean(axis=1)
            if vals.max()==0: continue
            cut=np.percentile(vals,100-top_pct)
            out.extend((chrom,i*bin_size,(i+1)*bin_size) for i in np.where(vals>cut)[0])
        bw.close(); return out

    rec=[]
    for f in tqdm(samples.file.tolist(), desc="Peak union"):
        rec.extend(top_bins(f))
    assert rec, "No top-bins."
    peaks = pr.PyRanges(pd.DataFrame(rec, columns=["Chromosome","Start","End"])).merge(slack=500)
    peaks = peaks[peaks.lengths()>=500]
    peak_df = peaks.df.reset_index(drop=True)
    peak_ids=[f"{c}:{s}-{e}" for c,s,e in zip(peak_df.Chromosome,peak_df.Start,peak_df.End)]
    log(">>> Consensus regions:", len(peak_df))

    peaks_by_chr={c:g for c,g in peak_df.groupby("Chromosome")}
    M=np.zeros((len(peak_df),len(samples)),dtype=np.float32)
    for j in tqdm(range(len(samples)), desc=f"Quantify {MARK}"):
        bw=pyBigWig.open(samples.iloc[j].file); cs=bw.chroms()
        for chrom,gdf in peaks_by_chr.items():
            if chrom not in cs: continue
            L=cs[chrom]; arr=np.nan_to_num(bw.values(chrom,0,L,numpy=True))
            st=gdf.Start.values.astype(int); en=np.minimum(gdf.End.values.astype(int),L)
            for i,s,e in zip(gdf.index,st,en):
                if e>s: M[i,j]=arr[s:e].mean()
        bw.close()
    cols=[f"{r.age}_r{r.rep}" for _,r in samples.iterrows()]
    M=pd.DataFrame(M,index=peak_ids,columns=cols)

    # --- normalize + 24M vs 3M differential ---
    logM=np.log2(M+1)
    qn=pd.DataFrame(quantile_transform(logM.values,axis=0,output_distribution="normal"),
                    index=M.index,columns=M.columns)
    def cols_for(age): return [c for c in qn.columns if c.startswith(age+"_")]
    c3,c24=cols_for("3M"),cols_for("24M")
    t_stat,p_val=stats.ttest_ind(qn[c24].values,qn[c3].values,axis=1)
    lfc=qn[c24].mean(1)-qn[c3].mean(1)
    fdr=multipletests(np.nan_to_num(p_val,nan=1.0),method="fdr_bh")[1]
    de=pd.DataFrame({"lfc_24m_vs_3m":lfc.values,"p_24m":p_val,"fdr_24m":fdr},index=qn.index)

    MIN_REGIONS,TOP_N=50,2000
    sig=de[(de.fdr_24m<0.05)&(de.lfc_24m_vs_3m.abs()>0.25)]; sel_mode="strict"
    if len(sig)<MIN_REGIONS:
        relaxed=de[(de.p_24m<0.01)&(de.lfc_24m_vs_3m.abs()>0.25)]
        if len(relaxed)>=MIN_REGIONS: sig,sel_mode=relaxed,"relaxed (p<0.01)"
        else: sig,sel_mode=de.sort_values("p_24m").head(TOP_N),f"fallback top-{TOP_N}"
    log(f">>> Differential mode: {sel_mode} | regions: {len(sig)}")

    # --- trajectory clustering 3M->12M->24M ---
    ages_order=[a for a in ["3M","12M","24M"] if cols_for(a)]
    means=pd.DataFrame({a:qn[cols_for(a)].mean(1) for a in ages_order}).loc[sig.index]
    Z=means.sub(means.mean(1),axis=0).div(means.std(1).replace(0,1),axis=0)
    loss_cl,gain_cl=[],[]
    if len(Z)>=4:
        nclu=min(6,max(2,len(Z)//50)); nclu=min(nclu,len(Z))
        km=KMeans(n_clusters=nclu,random_state=42,n_init=20).fit(Z.values)
        Z["cluster"]=km.labels_; prof=Z.groupby("cluster")[ages_order].mean()
        def mdown(p):
            sq=[p[a] for a in ages_order]
            return all(sq[i]>=sq[i+1] for i in range(len(sq)-1)) and p["24M"]<p["3M"] and p["24M"]<-0.25
        def mup(p):
            sq=[p[a] for a in ages_order]
            return all(sq[i]<=sq[i+1] for i in range(len(sq)-1)) and p["24M"]>p["3M"] and p["24M"]>0.25
        loss_cl=prof.index[prof.apply(mdown,axis=1)].tolist()
        gain_cl=prof.index[prof.apply(mup,axis=1)].tolist()
        persistent_idx=Z[Z.cluster.isin(loss_cl+gain_cl)].index
    else:
        Z["cluster"]=0; persistent_idx=Z.index
    if len(persistent_idx)==0: persistent_idx=sig.index
    log(f">>> Progressive LOSS cl {loss_cl}, GAIN cl {gain_cl}; regions->ABC: {len(persistent_idx)}")

    pers=peak_df.copy(); pers.index=peak_ids; pers=pers.loc[persistent_idx].copy()
    pers["direction"]=np.where(de.loc[persistent_idx,"lfc_24m_vs_3m"].values>0,"GAIN","LOSS")

    # --- GENCODE vM25 protein-coding TSS ---
    GTF="abc_inputs/gencode.vM25.annotation.gtf.gz"
    if not os.path.exists(GTF):
        subprocess.run(f"wget -q -O {GTF} "
            "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/"
            "release_M25/gencode.vM25.annotation.gtf.gz", shell=True, check=True)
    from gtfparse import read_gtf
    gtf=read_gtf(GTF); gtf=gtf.to_pandas() if hasattr(gtf,"to_pandas") else gtf
    g=gtf[(gtf["feature"]=="gene")&(gtf["gene_type"]=="protein_coding")&
          (gtf["seqname"].isin(CHROMS))].copy()
    g["tss"]=np.where(g.strand=="+",g.start,g.end)
    tss=g[["seqname","tss","gene_name","strand","gene_id"]].rename(columns={"seqname":"Chromosome"})
    log(">>> Protein-coding genes:", len(tss))

    # --- activity (mean 24M signal) + ABC power-law scoring ---
    act=M.loc[persistent_idx,c24].mean(axis=1).rename("activity").reset_index()
    act.columns=["peak_id","activity"]
    act["Chromosome"]=act.peak_id.str.split(":").str[0]
    act["Start"]=act.peak_id.str.split(":").str[1].str.split("-").str[0].astype(int)
    act["End"]  =act.peak_id.str.split(":").str[1].str.split("-").str[1].astype(int)
    GAMMA,SCALE,PSEUDO=1.024,5.9,1.0
    def contact(d): d=np.maximum(d,1000); return np.exp(SCALE)*np.power(d,-GAMMA)+PSEUDO
    WINDOW,THR=5_000_000,0.02
    enh_by_chr={c:gg.reset_index(drop=True) for c,gg in act.groupby("Chromosome")}
    preds=[]
    for _,gene in tqdm(tss.iterrows(),total=len(tss),desc="ABC scoring"):
        if gene.Chromosome not in enh_by_chr: continue
        E=enh_by_chr[gene.Chromosome]; mid=(E.Start+E.End)//2
        dist=np.abs(mid-gene.tss); mask=dist<=WINDOW
        if not mask.any(): continue
        sub=E.loc[mask].copy(); sub["distance"]=dist[mask].values
        sub["contact"]=contact(sub.distance.values); sub["AxC"]=sub.activity*sub.contact
        denom=sub.AxC.sum()
        if denom<=0: continue
        sub["ABC_score"]=sub.AxC/denom; keep=sub[sub.ABC_score>THR].copy()
        if len(keep)==0: continue
        keep["TargetGene"]=gene.gene_name; preds.append(keep)
    ABC=pd.concat(preds,ignore_index=True) if preds else pd.DataFrame()
    log(">>> ABC predictions above threshold:", len(ABC))

    aging_genes=set(); aging_dir={}
    if len(ABC):
        dmap=pers.set_index(pers.Chromosome+":"+pers.Start.astype(str)+"-"+pers.End.astype(str))["direction"]
        ABC["direction"]=ABC.peak_id.map(dmap.to_dict())
        ABC["lfc_24m_vs_3m"]=ABC.peak_id.map(de.lfc_24m_vs_3m.to_dict())
        ABC["fdr_24m"]=ABC.peak_id.map(de.fdr_24m.to_dict())
        gcol="TargetGene"
        best=(ABC.sort_values("ABC_score",ascending=False).drop_duplicates(gcol))
        best=best[~best[gcol].astype(str).str.contains(r"Rik\d*$",na=False)]
        aging_genes=set(best[gcol].dropna().astype(str).str.upper().unique())
        aging_dir=(ABC.dropna(subset=[gcol]).groupby(gcol)["direction"]
                   .agg(lambda x: x.mode()[0] if len(x.mode())>0 else x.iloc[0]).to_dict())
        aging_dir={k.upper():v for k,v in aging_dir.items()}
        best.to_csv(f"{ROOT}/tables/aging_ABC_targets.tsv", sep="\t", index=False)

    RES["aging"].update({
        "mark": MARK, "mark_active": MARK_IS_ACTIVE,
        "n_consensus": len(peak_df), "sel_mode": sel_mode,
        "n_sig_regions": int(len(sig)),
        "loss_clusters": loss_cl, "gain_clusters": gain_cl,
        "n_persistent": int(len(persistent_idx)),
        "n_abc_pairs": int(len(ABC)),
        "aging_genes": aging_genes, "aging_dir": aging_dir,
        "n_gain": sum(1 for v in aging_dir.values() if str(v).upper()=="GAIN"),
        "n_loss": sum(1 for v in aging_dir.values() if str(v).upper()=="LOSS"),
    })
    log(f">>> Aging K27me3 target genes: {len(aging_genes)} "
        f"(GAIN={RES['aging']['n_gain']}, LOSS={RES['aging']['n_loss']})")
    AGING_OK = True
    os.chdir(ROOT)
except Exception as e:
    os.chdir(ROOT)
    log(">>> PART B FAILED:", e); log(traceback.format_exc())

# ============================================================================
# PART C —  COMPARISON  (gene-set + pathway + direction concordance)
# ============================================================================
log("\n" + "#"*78)
log("# PART C — COMPARISON: aging epigenetic targets vs NMN-rescue program")
log("#"*78)

aging_genes  = RES["aging"].get("aging_genes", set())
aging_dir    = RES["aging"].get("aging_dir", {})
nmn_robust   = RES["nmn"].get("robust_genes", set())
nmn_union    = RES["nmn"].get("union_genes", set())
nmn_dir      = RES["nmn"].get("nmn_dir", {})
nmn_mito     = RES["nmn"].get("nmn_mito", set())
MARK         = RES["aging"].get("mark", "K27me3")
MARK_IS_ACTIVE = RES["aging"].get("mark_active", False)

# Prefer robust NMN set; fall back to union if robust too small
nmn_primary = nmn_robust if len(nmn_robust) >= 20 else nmn_union
nmn_primary_label = "robust(>=2 tissues)" if nmn_robust is nmn_primary else "union(all tissues)"

def overlap_stats(s1, s2, universe=UNIVERSE_N):
    inter = s1 & s2; k = len(inter)
    n1, n2 = len(s1), len(s2)
    p = hypergeom.sf(k-1, universe, n2, n1) if (k>=1 and n1 and n2) else 1.0
    jacc = k/len(s1|s2) if (s1|s2) else 0.0
    return inter, k, p, jacc

shared, k, pval, jacc = overlap_stats(aging_genes, nmn_primary)
RES["compare"]["gene_overlap"] = {
    "n_aging": len(aging_genes), "n_nmn": len(nmn_primary),
    "nmn_label": nmn_primary_label, "k": k, "p": pval, "jaccard": jacc,
    "shared": sorted(shared)}
log(f">>> Aging ({len(aging_genes)}) ∩ NMN {nmn_primary_label} ({len(nmn_primary)}): "
    f"{k} genes | Jaccard={jacc:.3f} | hypergeom p={pval:.2e}")

# Shared-gene direction table
shared_rows = []
for g in sorted(shared):
    ad = aging_dir.get(g, "NA")
    nd = nmn_dir.get(g, "NA")
    nd_lab = ("NMN_raises_old" if nd==1 else "NMN_lowers_old" if nd==-1 else "NA")
    shared_rows.append({"gene": g, f"aging_{MARK}_dir": ad,
                        "nmn_interaction_sign": nd, "nmn_effect": nd_lab})
shared_df = pd.DataFrame(shared_rows)
if len(shared_df):
    shared_df.to_csv(f"{ROOT}/tables/shared_genes_direction.tsv", sep="\t", index=False)
RES["compare"]["shared_df"] = shared_df

# Venn plot
try:
    from matplotlib_venn import venn2
    if len(aging_genes) and len(nmn_primary):
        plt.figure(figsize=(6,5))
        venn2([aging_genes, nmn_primary],
              set_labels=(f"Aging {MARK} targets", f"NMN rescue\n{nmn_primary_label}"),
              set_colors=("#e74c3c","#3498db"), alpha=0.6)
        plt.title("Gene-set overlap: epigenetic aging vs NMN-rescue")
        plt.tight_layout(); plt.savefig(f"{ROOT}/plots/venn_aging_vs_nmn.png",
                                        bbox_inches="tight"); plt.close()
        log(">>> Venn written.")
except Exception as e:
    log(">>> Venn skipped:", e)

# Enrichr on BOTH sets (apples-to-apples) -> shared significant terms
log(">>> Running Enrichr on aging targets...")
enr_aging = run_enrichr(sorted(aging_genes), "AGING")
log(">>> Running Enrichr on NMN-rescue targets...")
enr_nmn   = run_enrichr(sorted(nmn_primary), "NMN")

def sig_terms(df):
    if df is None or not len(df) or "Adjusted P-value" not in df.columns: return set()
    s = df[df["Adjusted P-value"] < ADJP_CUT]
    return set(zip(s["Library"], s["Term"].str.lower().str.strip()))

aging_terms = sig_terms(enr_aging)
nmn_terms   = sig_terms(enr_nmn)
shared_terms = aging_terms & nmn_terms
RES["compare"]["aging_sig_terms"] = len(aging_terms)
RES["compare"]["nmn_sig_terms"]   = len(nmn_terms)
RES["compare"]["shared_terms"]    = sorted(shared_terms)
log(f">>> Sig Enrichr terms: aging={len(aging_terms)} nmn={len(nmn_terms)} "
    f"shared={len(shared_terms)}")
if shared_terms:
    pd.DataFrame(sorted(shared_terms), columns=["Library","Term"]).to_csv(
        f"{ROOT}/tables/shared_significant_terms.tsv", sep="\t", index=False)

# Save per-set top sig terms for the report
def top_terms(df, n=15):
    if df is None or not len(df) or "Adjusted P-value" not in df.columns: return []
    s = df[df["Adjusted P-value"] < ADJP_CUT].sort_values("Adjusted P-value").head(n)
    return [(r["Library"], str(r["Term"])[:70], r["Adjusted P-value"]) for _,r in s.iterrows()]
RES["compare"]["aging_top_terms"] = top_terms(enr_aging)
RES["compare"]["nmn_top_terms"]   = top_terms(enr_nmn)

# Shared functional THEMES (keyword) across both gene programs
THEMES = {
    "Mitochondrial/Energy": r"mitochond|oxidative phosphoryl|electron transport|respiratory|ATP|NADH|TCA|tricarboxylic",
    "Chromatin/Epigenetic": r"chromatin|histone|methylation|acetylation|polycomb|prc2|epigenet|nucleosome",
    "Inflammaging/Immune":  r"inflamm|immune|interferon|cytokine|complement|nf-?kb|innate",
    "Synaptic/Plasticity":  r"synap|plasticity|neurotroph|learning|memory|spine|axon|dendrit",
    "Aging/Senescence":     r"senescen|aging|ageing|longevity|sasp|telomere",
    "Development/Morphogen":r"development|morphogen|differentiation|pattern|homeobox|hox|cell fate",
    "Signaling":            r"mtor|mapk|erk|pi3k|akt|camp|pka|wnt|hippo|insulin signaling|foxo",
}
def theme_counts(terms):
    out = {}
    for theme,pat in THEMES.items():
        out[theme] = sum(1 for (lib,t) in terms if re.search(pat, t, re.I))
    return out
RES["compare"]["aging_theme_counts"] = theme_counts(aging_terms)
RES["compare"]["nmn_theme_counts"]   = theme_counts(nmn_terms)
RES["compare"]["shared_theme_counts"]= theme_counts(shared_terms)

# Mitochondrial overlap between programs (curated core)
aging_mito = aging_genes & FALLBACK_MITO
RES["compare"]["aging_mito"] = sorted(aging_mito)
RES["compare"]["nmn_mito"]   = sorted(nmn_mito)
RES["compare"]["mito_shared"]= sorted(aging_mito & nmn_mito)
log(f">>> Mito core genes: aging={len(aging_mito)} nmn={len(nmn_mito)} "
    f"shared={len(aging_mito & nmn_mito)}")

# Direction concordance on shared genes (repressive-mark logic)
conc_summary = {"concordant":0, "discordant":0, "undetermined":0, "rows":[]}
for g in sorted(shared):
    ad = str(aging_dir.get(g,"NA")).upper(); nd = nmn_dir.get(g, None)
    verdict = "undetermined"
    if ad in ("GAIN","LOSS") and nd in (1,-1):
        # repressive K27me3: GAIN=more repression(expr down with age);
        # NMN concordant rescue = opposes that -> for GAIN expect NMN raise old (nd=+1);
        #                                          for LOSS expect NMN lower old (nd=-1)
        if not MARK_IS_ACTIVE:
            concordant = (ad=="GAIN" and nd==1) or (ad=="LOSS" and nd==-1)
        else:
            concordant = (ad=="GAIN" and nd==-1) or (ad=="LOSS" and nd==1)
        verdict = "concordant" if concordant else "discordant"
        conc_summary["concordant" if concordant else "discordant"] += 1
    else:
        conc_summary["undetermined"] += 1
    conc_summary["rows"].append((g, ad, nd, verdict))
RES["compare"]["concordance"] = conc_summary
if conc_summary["rows"]:
    pd.DataFrame(conc_summary["rows"],
        columns=["gene",f"aging_{MARK}_dir","nmn_sign","verdict"]).to_csv(
        f"{ROOT}/tables/direction_concordance.tsv", sep="\t", index=False)

# ============================================================================
# PART D —  DETAILED MASTER SUMMARY  ->  MASTER_SUMMARY.txt
# ============================================================================
def fmt_p(x):
    try: return f"{float(x):.2e}"
    except: return str(x)
bar="="*80; dash="-"*80
S=[]; w=lambda *a: S.append(" ".join(str(x) for x in a))
now=datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

w("#"*80)
w("#  MASTER SUMMARY — EPIGENETIC AGING (GSE190102) vs NMN-RESCUE (GSE85718)")
w("#"*80)
w(f"Generated      : {now}")
w(f"Workspace      : {ROOT}")
w(f"NMN stage ran  : {NMN_OK}   |   Aging stage ran : {AGING_OK}")
w("")

# ---- HEADLINE / KEY FINDINGS -------------------------------------------------
w(bar); w("KEY FINDINGS  (auto-synthesized — the important / significant results)"); w(bar)
go = RES["compare"].get("gene_overlap", {})
if go:
    sig_word = ("SIGNIFICANT" if go["p"]<0.05 else "not significant")
    w(f"[1] Gene-level overlap is {sig_word}: {go['k']} shared genes between "
      f"{go['n_aging']} aging {MARK} targets and {go['n_nmn']} NMN-rescue "
      f"({go['nmn_label']}) genes.")
    w(f"      Jaccard={go['jaccard']:.3f} | hypergeometric p={fmt_p(go['p'])} "
      f"(universe={UNIVERSE_N}).")
    if go["shared"]:
        w("      Shared genes: " + ", ".join(go["shared"][:40]) +
          (" ..." if len(go["shared"])>40 else ""))
st = RES["compare"].get("shared_terms", [])
w(f"[2] Pathway overlap: {RES['compare'].get('aging_sig_terms',0)} aging vs "
  f"{RES['compare'].get('nmn_sig_terms',0)} NMN significant terms (adjP<{ADJP_CUT}); "
  f"{len(st)} SHARED significant terms.")
if st:
    for lib,term in st[:12]:
        w(f"      - [{lib.split('_')[0]}] {term}")
sh_theme = RES["compare"].get("shared_theme_counts", {})
hot = sorted([(k,v) for k,v in sh_theme.items() if v>0], key=lambda x:-x[1])
if hot:
    w("[3] Shared functional THEMES (both programs significant): " +
      "; ".join(f"{k}({v})" for k,v in hot))
ms = RES["compare"].get("mito_shared", [])
w(f"[4] Mitochondrial/energy core overlap: aging={len(RES['compare'].get('aging_mito',[]))}, "
  f"NMN={len(RES['compare'].get('nmn_mito',[]))}, shared={len(ms)}"
  + (" -> " + ", ".join(ms) if ms else ""))
cs = RES["compare"].get("concordance", {})
if cs and (cs["concordant"]+cs["discordant"]):
    w(f"[5] Direction concordance on shared genes: "
      f"{cs['concordant']} concordant, {cs['discordant']} discordant, "
      f"{cs['undetermined']} undetermined (repressive-mark logic).")
w("")
w(dash); w("Full detail follows."); w(dash); w("")

# ---- SECTION 1: NMN program --------------------------------------------------
w(bar); w("1. NMN-RESCUE TRANSCRIPTOMIC PROGRAM  (GSE85718)"); w(bar)
if NMN_OK:
    w("Study: 12-month oral NMN (300 mg/kg/day) vs aging in C57BL/6N; tissues "
      "Skeletal muscle / Liver / WAT; ages 6M/12M; model expr ~ age*treatment.")
    w(f"Probe->gene: {RES['nmn']['n_probes_in']} probes -> {RES['nmn']['n_genes']} genes.")
    w(f"Normalization: {RES['nmn']['norm_note']}")
    w("Design (tissue,age,treatment -> n):")
    for kk,vv in RES["nmn"]["design"].items(): w(f"   {kk}: {vv}")
    w("")
    w("Differential significance (genes at FDR<0.05):")
    w(f"   {'tissue':16s} {'age(Ctrl)':>10s} {'NMN@6M':>8s} {'interaction':>12s}")
    for t,c in RES["nmn"]["de_counts"].items():
        w(f"   {t:16s} {c['age_Control']:>10d} {c['NMN_at6M']:>8d} {c['interaction']:>12d}")
    w("")
    w("NMN-rescue genes (age-sig in Control + sign-reversal interaction):")
    for t in ["Skeletal_muscle","Liver","WAT"]:
        w(f"   {t:16s}: {RES['nmn']['rescue_counts'][t]:4d}  [{RES['nmn']['rescue_mode'][t]}]")
    w(f"Cross-tissue robust (>=2 tissues): {len(RES['nmn']['robust_genes'])}")
    w(f"Rescued in ALL 3 tissues        : {len(RES['nmn']['genes_all3'])}"
      + ("  (" + ", ".join(sorted(RES['nmn']['genes_all3'])) + ")"
         if RES['nmn']['genes_all3'] else ""))
    rd = RES["nmn"]["robust_detail"]
    if len(rd):
        w("\nTop robust rescue genes (by |mean interaction coef|):")
        for _,r in rd.reindex(rd.mean_coef_interaction.abs().sort_values(ascending=False).index).head(25).iterrows():
            w(f"   {r['gene']:14s} n_tis={int(r['n_tissues'])} "
              f"meanInt={r['mean_coef_interaction']:+.3f} ageCtrl={r['mean_coef_age_Control']:+.3f}")
    w(f"\nNMN robust/union genes annotated mitochondrial (curated core): "
      f"{len(RES['nmn']['nmn_mito'])}"
      + ("  -> " + ", ".join(sorted(RES['nmn']['nmn_mito'])) if RES['nmn']['nmn_mito'] else ""))
else:
    w("  (NMN stage failed — see run log)")
w("")

# ---- SECTION 2: Aging program ------------------------------------------------
w(bar); w("2. EPIGENETIC AGING PROGRAM  (GSE190102 ABC)"); w(bar)
if AGING_OK:
    a=RES["aging"]
    w(f"Histone mark detected: {a['mark']}  (active-enhancer-type = {a['mark_active']})")
    if not a["mark_active"]:
        w("  CAVEAT: repressive mark — 'activity'/'ABC' reflect mark intensity at")
        w("  Polycomb/heterochromatin loci. GAIN ~ increased repression; LOSS ~ de-repression.")
    w(f"Consensus regions      : {a['n_consensus']}")
    w(f"Differential selection : {a['sel_mode']}  -> {a['n_sig_regions']} age regions (24M vs 3M)")
    w(f"Trajectory clusters    : progressive LOSS={a['loss_clusters']} GAIN={a['gain_clusters']}")
    w(f"Regions carried to ABC : {a['n_persistent']}")
    w(f"ABC region-gene pairs  : {a['n_abc_pairs']}")
    w(f"Target genes           : {len(a['aging_genes'])} "
      f"(GAIN={a['n_gain']}, LOSS={a['n_loss']})")
    if a["aging_genes"]:
        w("  Example targets: " + ", ".join(sorted(a["aging_genes"])[:40]) +
          (" ..." if len(a["aging_genes"])>40 else ""))
    am=RES["compare"].get("aging_mito",[])
    w(f"  Aging targets in mito core: {len(am)}"
      + ("  -> " + ", ".join(am) if am else ""))
else:
    w("  (Aging stage failed — see run log)")
w("")

# ---- SECTION 3: Gene-set overlap --------------------------------------------
w(bar); w("3. GENE-SET OVERLAP (aging epigenetic targets vs NMN-rescue)"); w(bar)
if go:
    w(f"Aging {MARK} targets            : {go['n_aging']}")
    w(f"NMN-rescue [{go['nmn_label']}] : {go['n_nmn']}")
    w(f"Intersection                   : {go['k']} genes")
    w(f"Jaccard index                  : {go['jaccard']:.4f}")
    w(f"Hypergeometric p-value         : {fmt_p(go['p'])}")
    sdf=RES["compare"].get("shared_df")
    if sdf is not None and len(sdf):
        w("\nShared genes with direction (aging mark dir | NMN interaction effect):")
        for _,r in sdf.iterrows():
            w(f"   {r['gene']:16s} aging={str(r.get(f'aging_{MARK}_dir','NA')):6s}"
              f"  nmn={r.get('nmn_effect','NA')}")
else:
    w("  (overlap not computed)")
w("")

# ---- SECTION 4: Pathway / enrichment comparison -----------------------------
w(bar); w("4. PATHWAY / ENRICHMENT COMPARISON (Enrichr, adjP<0.05)"); w(bar)
w(f"Aging significant terms : {RES['compare'].get('aging_sig_terms',0)}")
w(f"NMN significant terms   : {RES['compare'].get('nmn_sig_terms',0)}")
w(f"SHARED significant terms: {len(RES['compare'].get('shared_terms',[]))}")
if RES["compare"].get("shared_terms"):
    w("\nShared significant terms:")
    for lib,term in RES["compare"]["shared_terms"][:40]:
        w(f"   [{lib}] {term}")
w("\nTop aging enriched terms:")
for lib,term,p in RES["compare"].get("aging_top_terms",[]):
    w(f"   [{lib.split('_')[0]}] {term}  (adjP={fmt_p(p)})")
w("\nTop NMN enriched terms:")
for lib,term,p in RES["compare"].get("nmn_top_terms",[]):
    w(f"   [{lib.split('_')[0]}] {term}  (adjP={fmt_p(p)})")
w("")
w("Functional theme counts (significant terms matching each theme):")
w(f"   {'theme':24s} {'aging':>7s} {'NMN':>7s} {'shared':>7s}")
for th in THEMES:
    w(f"   {th:24s} "
      f"{RES['compare']['aging_theme_counts'].get(th,0):>7d} "
      f"{RES['compare']['nmn_theme_counts'].get(th,0):>7d} "
      f"{RES['compare']['shared_theme_counts'].get(th,0):>7d}")
w("")

# ---- SECTION 5: Mito + direction concordance --------------------------------
w(bar); w("5. MITOCHONDRIAL / ENERGY & DIRECTION CONCORDANCE"); w(bar)
w(f"Aging mito-core genes : {len(RES['compare'].get('aging_mito',[]))}")
w(f"NMN mito-core genes   : {len(RES['compare'].get('nmn_mito',[]))}")
w(f"Shared mito-core genes: {len(RES['compare'].get('mito_shared',[]))}"
  + ("  -> " + ", ".join(RES['compare']['mito_shared']) if RES['compare'].get('mito_shared') else ""))
cs=RES["compare"].get("concordance",{})
if cs:
    w(f"\nDirection concordance on shared genes "
      f"({'repressive' if not MARK_IS_ACTIVE else 'active'}-mark logic):")
    w(f"   concordant={cs['concordant']}  discordant={cs['discordant']}  "
      f"undetermined={cs['undetermined']}")
    for g,ad,nd,v in cs.get("rows",[])[:40]:
        w(f"   {g:16s} aging={ad:6s} nmn_sign={str(nd):>4s} -> {v}")
w("")

# ---- SECTION 6: Interpretation ----------------------------------------------
w(bar); w("6. INTERPRETATION NOTES"); w(bar)
w(textwrap.fill(
  "These two studies sit in different tissue contexts (neuron epigenome vs "
  "metabolic-tissue transcriptome), so low EXACT-gene overlap is expected; "
  "pathway/hallmark concordance is the more meaningful signal. A significant "
  "hypergeometric overlap or shared mito/energy/chromatin/inflammaging terms "
  "would support that long-term NMN engages (or opposes) the same core aging "
  "programs that are epigenetically remodelled in aging neurons.", width=80))
w("")
w(textwrap.fill(
  f"For the repressive mark {MARK}: GAIN with age implies increased Polycomb "
  "repression (expression tends down); a biologically coherent NMN rescue on "
  "such a gene would RAISE its old-age expression (positive interaction sign). "
  "The concordance tallies above operationalize this expectation, but with "
  "n=4/cell (NMN) and few ChIP replicates, treat all results as "
  "hypothesis-generating.", width=80))
w("")
w(bar); w("7. RUN LOG"); w(bar)
S.extend("  "+l for l in RUN_LOG)
w(""); w("#"*80); w("#  END OF MASTER SUMMARY"); w("#"*80)

SUMMARY_PATH = f"{ROOT}/MASTER_SUMMARY.txt"
with open(SUMMARY_PATH,"w") as fh: fh.write("\n".join(S)+"\n")

print("\n>>> MASTER SUMMARY written to:", SUMMARY_PATH)
print(">>> Preview (first 70 lines):\n")
print("\n".join(S[:70]))
print("\n" + "="*60)
print("DONE — outputs under:", ROOT)
print("  MASTER_SUMMARY.txt   (detailed all-in-one)")
print("  tables/  plots/")
subprocess.run(f"find {ROOT} -maxdepth 2 -type f | sort", shell=True)


##############################################################################
# PART A — GSE85718 NMN aging microarray  (building rescue gene program)
##############################################################################
>>> Downloading & parsing GSE85718 (SOFT family file)...
>>> Raw probe matrix: (25697, 48)
>>> Sample design: {('Liver', '12M', 'Control'): 4, ('Liver', '12M', 'NMN'): 4, ('Liver', '6M', 'Control'): 4, ('Liver', '6M', 'NMN'): 4, ('Skeletal_muscle', '12M', 'Control'): 4, ('Skeletal_muscle', '12M', 'NMN'): 4, ('Skeletal_muscle', '6M', 'Control'): 4, ('Skeletal_muscle', '6M', 'NMN'): 4, ('WAT', '12M', 'Control'): 4, ('WAT', '12M', 'NMN'): 4, ('WAT', '6M', 'Control'): 4, ('WAT', '6M', 'NMN'): 4}
>>> Gene-level matrix: (18120, 48) (from 25697 probes)
>>> Skeletal_muscle  sig age:    0  NMN@6M:    0  interaction:    0
>>> Liver            sig age:    0  NMN@6M:    0  interaction:    0
>>> WAT              sig age:    0  NMN@6M:    0  interaction:    0
>>> Skeletal

Peak union:   0%|          | 0/15 [00:00<?, ?it/s]

>>> Consensus regions: 366721


Quantify K27me3:   0%|          | 0/15 [00:00<?, ?it/s]

>>> Differential mode: relaxed (p<0.01) | regions: 9752
>>> Progressive LOSS cl [1, 4], GAIN cl [0, 3]; regions->ABC: 6402
>>> Protein-coding genes: 21674


ABC scoring:   0%|          | 0/21674 [00:00<?, ?it/s]

>>> ABC predictions above threshold: 427099
>>> Aging K27me3 target genes: 21155 (GAIN=9178, LOSS=12470)

##############################################################################
# PART C — COMPARISON: aging epigenetic targets vs NMN-rescue program
##############################################################################
>>> Aging (21155) ∩ NMN robust(>=2 tissues) (35): 23 genes | Jaccard=0.001 | hypergeom p=nan
>>> Venn written.
>>> Running Enrichr on aging targets...
  [AGING] GO_Biological_Process_2023                 terms= 2846  sig=225
  [AGING] KEGG_2019_Mouse                            terms=  279  sig=102
  [AGING] Reactome_2022                              terms= 1208  sig=149
  [AGING] MSigDB_Hallmark_2020                       terms=   50  sig=5
  [AGING] MGI_Mammalian_Phenotype_Level_4_2021       terms= 3591  sig=8
>>> Running Enrichr on NMN-rescue targets...
  [NMN] GO_Biological_Process_2023                 terms=  165  sig=0
  [NMN] KEGG_2019_Mouse           

CompletedProcess(args='find /content/AGING_vs_NMN_MASTER -maxdepth 2 -type f | sort', returncode=0)

# DOWNSTREAM

In [ ]:
# ============================================================================
# DOWNSTREAM DEEP-DIVE  —  consumes ONLY the outputs of 088eva.py
#   (AGING GSE190102 K27me3  vs  NMN-RESCUE GSE85718  master pipeline)
# Produces one very detailed DOWNSTREAM_DEEP_DIVE.txt at the end.
# Run as a single Colab cell AFTER the master pipeline has finished.
# ============================================================================

# ---------- 0. Deps ----------
import subprocess
subprocess.run("pip -q install gseapy requests seaborn matplotlib-venn",
               shell=True, check=False)

import os, re, glob, time, textwrap, datetime, warnings, traceback
from io import StringIO
import numpy as np, pandas as pd
from scipy.stats import hypergeom
import matplotlib.pyplot as plt
import seaborn as sns
import requests
warnings.filterwarnings("ignore")
sns.set(style="whitegrid"); plt.rcParams["figure.dpi"] = 140

try:
    import gseapy as gp; HAVE_GSEAPY = True
except Exception as e:
    print("gseapy unavailable:", e); HAVE_GSEAPY = False

# ---------- Paths / config ----------
MASTER_OUT = "/content/AGING_vs_NMN_MASTER"      # <- master pipeline root
# allow a Drive fallback if the local copy is gone
if not os.path.isdir(MASTER_OUT):
    for cand in glob.glob("/content/drive/MyDrive/**/AGING_vs_NMN_MASTER",
                          recursive=True):
        MASTER_OUT = cand; break
TABLES = f"{MASTER_OUT}/tables"
OUT    = f"{MASTER_OUT}/downstream"
for d in [OUT, f"{OUT}/plots", f"{OUT}/tables", f"{OUT}/enrichr"]:
    os.makedirs(d, exist_ok=True)

ADJP_CUT     = 0.05
UNIVERSE_N   = 20000
STRING_CONF  = 0.4          # 400/1000
SPECIES_ID   = 10090        # mouse
MARK         = "K27me3"     # repressive mark from the master run (auto-checked below)

ENRICHR_LIBS = ["GO_Biological_Process_2023", "KEGG_2019_Mouse",
                "Reactome_2022", "MSigDB_Hallmark_2020",
                "MGI_Mammalian_Phenotype_Level_4_2021"]

# Same curated mito/OXPHOS core used upstream
FALLBACK_MITO = {
    "NDUFA1","NDUFA2","NDUFA9","NDUFB8","NDUFS1","NDUFS3","NDUFV1","SDHA","SDHB",
    "UQCRC1","UQCRC2","CYC1","COX4I1","COX5A","COX5B","COX6A1","COX7A1","COX8B",
    "ATP5A1","ATP5B","ATP5C1","ATP5O","ATP5J","CS","IDH3A","SUCLA2","FH1","MDH2",
    "ACO2","OGDH","PDHA1","PDHB","DLAT","CPT1B","CPT2","ACADM","ACADL","ACADVL",
    "HADHA","HADHB","ECI1","SLC25A4","SLC25A20","TFAM","NRF1","PPARGC1A","ESRRA",
    "MFN1","MFN2","OPA1","DNM1L","PINK1","PRKN","SOD2","TFB1M","TFB2M","POLG",
    "MRPL12","MRPS12","TIMM23","TOMM20","VDAC1","VDAC2","CYCS","UCP2","UCP3",
}

THEMES = {
    "Mitochondrial/Energy": r"mitochond|oxidative phosphoryl|electron transport|respiratory|ATP|NADH|TCA|tricarboxylic",
    "Chromatin/Epigenetic": r"chromatin|histone|methylation|acetylation|polycomb|prc2|epigenet|nucleosome",
    "Inflammaging/Immune":  r"inflamm|immune|interferon|cytokine|complement|nf-?kb|innate",
    "Synaptic/Plasticity":  r"synap|plasticity|neurotroph|learning|memory|spine|axon|dendrit",
    "Aging/Senescence":     r"senescen|aging|ageing|longevity|sasp|telomere",
    "Development/Morphogen":r"development|morphogen|differentiation|pattern|homeobox|hox|cell fate",
    "Signaling":            r"mtor|mapk|erk|pi3k|akt|camp|pka|wnt|hippo|insulin signaling|foxo",
}

RUN_LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); RUN_LOG.append(s)

RES = {}    # everything we'll fold into the report

log("#"*78)
log("# DOWNSTREAM DEEP-DIVE — using only 088eva.py master outputs")
log("# Source:", MASTER_OUT)
log("#"*78)

# ---------- 1. Load master tables (safe) ----------
def safe_read(name):
    path = f"{TABLES}/{name}"
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, sep="\t")
            log(f">>> loaded {name:38s} : {len(df):5d} rows, cols={list(df.columns)}")
            return df
        except Exception as e:
            log(f">>> FAILED to read {name}: {e}")
    else:
        log(f">>> MISSING {name}")
    return pd.DataFrame()

shared_dir   = safe_read("shared_genes_direction.tsv")
concord      = safe_read("direction_concordance.tsv")
shared_terms = safe_read("shared_significant_terms.tsv")
nmn_robust   = safe_read("NMN_robust_detail.tsv")
aging_abc    = safe_read("aging_ABC_targets.tsv")

# normalize a couple of column-name conveniences
def upper_col(df, col):
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper().str.strip()
    return df
for df in (shared_dir, concord, nmn_robust):
    if len(df): upper_col(df, "gene")
if len(aging_abc) and "TargetGene" in aging_abc.columns:
    aging_abc["TargetGene"] = aging_abc["TargetGene"].astype(str).str.upper().str.strip()

# detect aging-direction column name (e.g. aging_K27me3_dir)
aging_dir_col = next((c for c in shared_dir.columns
                      if c.startswith("aging_") and c.endswith("_dir")), None)
if aging_dir_col:
    m = re.search(r"aging_(.+)_dir", aging_dir_col)
    if m: MARK = m.group(1)
log(f">>> Mark inferred from tables: {MARK}")

shared_genes = (shared_dir["gene"].dropna().astype(str).str.upper().unique().tolist()
                if "gene" in shared_dir.columns else [])
RES["n_shared"] = len(shared_genes)
log(f">>> Shared genes available for downstream: {len(shared_genes)}")

if not len(shared_dir) and not len(aging_abc) and not len(nmn_robust):
    raise SystemExit("No master-pipeline tables found — run 088eva.py first "
                     f"(expected under {TABLES}).")

# ---------- Robust Enrichr wrapper ----------
def run_enrichr(gene_list, tag, libs=ENRICHR_LIBS):
    if not HAVE_GSEAPY:
        log(f"  [{tag}] Enrichr skipped (no gseapy)."); return pd.DataFrame()
    gl = list(dict.fromkeys([str(g).upper() for g in gene_list if str(g).strip()]))
    if len(gl) < 5:
        log(f"  [{tag}] Enrichr skipped (<5 genes: {len(gl)})."); return pd.DataFrame()
    if len(gl) > 2000: gl = gl[:2000]
    frames = []
    for lib in libs:
        ok = False
        for attempt in range(3):
            try:
                e = gp.enrichr(gene_list=gl, gene_sets=lib,
                               organism="mouse", outdir=None)
                cand = e.results.copy()
                if "Adjusted P-value" in cand.columns and len(cand):
                    cand["Library"] = lib; frames.append(cand)
                    nsig = int((cand["Adjusted P-value"] < ADJP_CUT).sum())
                    log(f"  [{tag}] {lib:42s} terms={len(cand):5d} sig={nsig}")
                    ok = True; break
                raise ValueError("no Adjusted P-value column")
            except Exception as ex:
                time.sleep(3*(attempt+1))
        if not ok:
            log(f"  [{tag}] {lib:42s} FAILED after retries.")
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if len(out):
        out.to_csv(f"{OUT}/enrichr/{tag}_ALL.tsv", sep="\t", index=False)
    return out

def fmt_p(x):
    try: return f"{float(x):.2e}"
    except: return str(x)

# ============================================================================
# 2. PRIORITIZED VALIDATION SHORTLIST
# ============================================================================
log("\n### 2. PRIORITIZED VALIDATION SHORTLIST ###")
shortlist = pd.DataFrame()
try:
    if len(shared_dir):
        sl = shared_dir.copy()
        if len(nmn_robust):
            keep = [c for c in ["gene","mean_coef_interaction","n_tissues",
                                "mean_coef_age_Control"] if c in nmn_robust.columns]
            sl = sl.merge(nmn_robust[keep], on="gene", how="left")
        if len(aging_abc) and "ABC_score" in aging_abc.columns:
            abc_best = (aging_abc.sort_values("ABC_score", ascending=False)
                                 .drop_duplicates("TargetGene"))
            acols = [c for c in ["TargetGene","ABC_score","direction","lfc_24m_vs_3m",
                                 "fdr_24m"] if c in abc_best.columns]
            sl = sl.merge(abc_best[acols], left_on="gene", right_on="TargetGene",
                          how="left").drop(columns=["TargetGene"], errors="ignore")
        if len(concord) and "verdict" in concord.columns:
            sl = sl.merge(concord[["gene","verdict"]].drop_duplicates("gene"),
                          on="gene", how="left")
        sl["is_mito"] = sl["gene"].isin(FALLBACK_MITO)

        # composite priority score
        sl["priority"] = 0.0
        if "verdict" in sl.columns:
            sl.loc[sl["verdict"]=="concordant", "priority"] += 3
        if "n_tissues" in sl.columns:
            sl["priority"] += (pd.to_numeric(sl["n_tissues"],errors="coerce")
                               .fillna(0).clip(upper=3))
        if "ABC_score" in sl.columns and sl["ABC_score"].notna().any():
            sl["priority"] += pd.to_numeric(sl["ABC_score"],errors="coerce").rank(pct=True).fillna(0)*2
        if "mean_coef_interaction" in sl.columns and sl["mean_coef_interaction"].notna().any():
            sl["priority"] += pd.to_numeric(sl["mean_coef_interaction"],errors="coerce").abs().rank(pct=True).fillna(0)*2
        sl["priority"] += sl["is_mito"].astype(int)*1

        sl = sl.sort_values("priority", ascending=False).reset_index(drop=True)
        sl.to_csv(f"{OUT}/tables/validation_shortlist.tsv", sep="\t", index=False)
        shortlist = sl
        show = [c for c in ["gene","priority","verdict","n_tissues",
                            "mean_coef_interaction","ABC_score","is_mito"]
                if c in sl.columns]
        log(">>> Top 15 prioritized genes:")
        log(sl[show].head(15).to_string(index=False))
    else:
        log(">>> No shared_genes_direction table — shortlist skipped.")
except Exception as e:
    log(">>> Shortlist FAILED:", e); log(traceback.format_exc())
RES["shortlist"] = shortlist

# ============================================================================
# 3. ENRICHMENT ON SHARED GENES ONLY
# ============================================================================
log("\n### 3. ENRICHR ON SHARED GENES ###")
enr_shared = run_enrichr(shared_genes, "SHARED")
sig_shared = pd.DataFrame()
if len(enr_shared):
    sig_shared = enr_shared[enr_shared["Adjusted P-value"] < ADJP_CUT].copy()
    sig_shared.to_csv(f"{OUT}/tables/enrichr_shared_sig.tsv", sep="\t", index=False)
    log(f">>> Significant terms (shared genes): {len(sig_shared)}")
    # barplot of top terms
    try:
        if len(sig_shared):
            d = sig_shared.sort_values("Adjusted P-value").head(15).copy()
            d["mlog10"] = -np.log10(d["Adjusted P-value"].clip(lower=1e-300))
            d = d.sort_values("mlog10")
            plt.figure(figsize=(9, max(3,0.45*len(d))))
            sns.barplot(x="mlog10", y="Term", data=d, color="#7b52ab")
            plt.xlabel("-log10(adjusted P)"); plt.ylabel("")
            plt.title(f"Shared-gene enrichment (top terms)")
            plt.tight_layout(); plt.savefig(f"{OUT}/plots/enrichr_shared.png",
                                            bbox_inches="tight"); plt.close()
            log(">>> Shared-gene barplot written.")
    except Exception as e:
        log(">>> Shared barplot skipped:", e)
RES["sig_shared"] = sig_shared

# ============================================================================
# 4. CONCORDANT vs DISCORDANT ENRICHMENT
# ============================================================================
log("\n### 4. CONCORDANT vs DISCORDANT ENRICHMENT ###")
split_enr = {}
if len(concord) and "verdict" in concord.columns:
    for verdict in ["concordant","discordant"]:
        genes_v = concord.loc[concord["verdict"]==verdict, "gene"].dropna().tolist()
        if len(genes_v) < 8:
            log(f">>> {verdict}: only {len(genes_v)} genes — Enrichr skipped.")
            split_enr[verdict] = pd.DataFrame(); continue
        e = run_enrichr(genes_v, verdict.upper(),
                        libs=["GO_Biological_Process_2023","KEGG_2019_Mouse",
                              "Reactome_2022"])
        s = (e[e["Adjusted P-value"]<ADJP_CUT] if len(e) else pd.DataFrame())
        if len(s):
            s.to_csv(f"{OUT}/tables/enrichr_{verdict}_sig.tsv", sep="\t", index=False)
        split_enr[verdict] = s
        log(f">>> {verdict}: {len(genes_v)} genes -> {len(s)} sig terms")
else:
    log(">>> No concordance table — split enrichment skipped.")
RES["split_enr"] = split_enr

# ============================================================================
# 5. MITOCHONDRIAL DEEP DIVE
# ============================================================================
log("\n### 5. MITOCHONDRIAL DEEP DIVE ###")
shared_mito = sorted(set(shared_genes) & FALLBACK_MITO)
mito_df = pd.DataFrame()
RES["shared_mito"] = shared_mito
log(f">>> Shared mitochondrial genes: {len(shared_mito)}"
    + ("  -> " + ", ".join(shared_mito) if shared_mito else ""))
# hypergeometric: are mito genes enriched among shared set?
mito_p = np.nan
if len(shared_genes):
    k = len(shared_mito); n_mito = len(FALLBACK_MITO); n_list = len(shared_genes)
    mito_p = hypergeom.sf(k-1, UNIVERSE_N, n_mito, n_list) if k>=1 else 1.0
    log(f">>> Mito hypergeometric p among shared set: {fmt_p(mito_p)}")
RES["mito_p"] = mito_p
if shared_mito and len(shared_dir):
    mito_df = shared_dir[shared_dir["gene"].isin(shared_mito)].copy()
    if len(nmn_robust):
        keep = [c for c in ["gene","mean_coef_interaction","n_tissues"] if c in nmn_robust.columns]
        mito_df = mito_df.merge(nmn_robust[keep], on="gene", how="left")
    mito_df.to_csv(f"{OUT}/tables/shared_mitochondrial_genes.tsv", sep="\t", index=False)
RES["mito_df"] = mito_df

# ============================================================================
# 6. STRING NETWORK + HUBS ON SHARED GENES
# ============================================================================
log("\n### 6. STRING PPI ON SHARED GENES ###")
hubs = pd.DataFrame(); string_net = pd.DataFrame()
RES["string_edges"] = 0; RES["string_nodes"] = 0
try:
    if len(shared_genes) >= 3:
        query = shared_genes[:200]
        url = "https://string-db.org/api/tsv/network"
        params = {"identifiers": "%0d".join(query), "species": SPECIES_ID,
                  "required_score": int(STRING_CONF*1000),
                  "caller_identity": "aging_nmn_downstream"}
        r = requests.post(url, data=params, timeout=120); r.raise_for_status()
        string_net = pd.read_csv(StringIO(r.text), sep="\t")
        if len(string_net) and {"preferredName_A","preferredName_B"}.issubset(string_net.columns):
            string_net.to_csv(f"{OUT}/tables/string_shared_network.tsv", sep="\t", index=False)
            deg = pd.concat([string_net.preferredName_A,
                             string_net.preferredName_B]).value_counts()
            hubs = pd.DataFrame({"gene": deg.index.astype(str), "degree": deg.values})
            hubs["is_mito"] = hubs.gene.str.upper().isin(FALLBACK_MITO)
            hubs.to_csv(f"{OUT}/tables/string_shared_hubs.tsv", sep="\t", index=False)
            RES["string_edges"] = int(len(string_net)); RES["string_nodes"] = int(deg.size)
            log(f">>> STRING edges={len(string_net)} nodes={deg.size}")
            log(">>> Top hubs:", ", ".join(hubs.head(10).gene.tolist()))
            th = hubs.head(20).sort_values("degree")
            plt.figure(figsize=(8, max(3,0.4*len(th))))
            sns.barplot(x="degree", y="gene", data=th, color="#4caf72")
            plt.title(f"Top hubs — shared-gene STRING network ({MARK})")
            plt.xlabel("interaction degree"); plt.ylabel("")
            plt.tight_layout(); plt.savefig(f"{OUT}/plots/string_shared_hubs.png",
                                            bbox_inches="tight"); plt.close()
        else:
            log(">>> STRING returned no usable edges at this confidence.")
    else:
        log(">>> Too few shared genes for STRING.")
except Exception as e:
    log(">>> STRING query FAILED:", e)
RES["hubs"] = hubs

# ============================================================================
# 7. VISUALS — aging effect vs NMN rescue strength
# ============================================================================
log("\n### 7. COEFFICIENT COMPARISON PLOTS ###")
try:
    if len(shared_dir) and len(nmn_robust) and "mean_coef_interaction" in nmn_robust.columns:
        pdf = shared_dir.merge(nmn_robust[["gene","mean_coef_interaction"]],
                               on="gene", how="inner")
        # bring in aging lfc if available
        if len(aging_abc) and {"TargetGene","lfc_24m_vs_3m"}.issubset(aging_abc.columns):
            lfc_best = (aging_abc.sort_values("ABC_score",ascending=False)
                        if "ABC_score" in aging_abc.columns else aging_abc)
            lfc_best = lfc_best.drop_duplicates("TargetGene")[["TargetGene","lfc_24m_vs_3m"]]
            pdf = pdf.merge(lfc_best, left_on="gene", right_on="TargetGene",
                            how="left").drop(columns=["TargetGene"], errors="ignore")
        if "lfc_24m_vs_3m" in pdf.columns and pdf["lfc_24m_vs_3m"].notna().any():
            plt.figure(figsize=(7,6))
            hue = aging_dir_col if aging_dir_col in pdf.columns else None
            sns.scatterplot(data=pdf, x="lfc_24m_vs_3m", y="mean_coef_interaction",
                            hue=hue, palette={"GAIN":"#e74c3c","LOSS":"#3498db"}
                            if hue else None, s=60, edgecolor="k", linewidth=0.3)
            for _,r in pdf.iterrows():
                plt.annotate(r["gene"], (r["lfc_24m_vs_3m"], r["mean_coef_interaction"]),
                             fontsize=6, alpha=0.7)
            plt.axhline(0, color="gray", ls="--"); plt.axvline(0, color="gray", ls="--")
            plt.xlabel(f"Aging {MARK} LFC (24M vs 3M)")
            plt.ylabel("NMN interaction coef (mean)")
            plt.title("Shared genes: aging mark change vs NMN rescue direction")
            plt.tight_layout(); plt.savefig(f"{OUT}/plots/shared_lfc_vs_nmn_coef.png",
                                            bbox_inches="tight"); plt.close()
            log(">>> Scatter (aging LFC vs NMN coef) written.")
        # priority bar of top shortlist genes
        if len(shortlist):
            tops = shortlist.head(20).sort_values("priority")
            plt.figure(figsize=(8, max(3,0.4*len(tops))))
            colors = ["#e74c3c" if m else "#3498db" for m in tops.get("is_mito", [False]*len(tops))]
            plt.barh(tops["gene"], tops["priority"], color=colors)
            plt.xlabel("priority score"); plt.title("Top validation candidates (red=mito)")
            plt.tight_layout(); plt.savefig(f"{OUT}/plots/shortlist_priority.png",
                                            bbox_inches="tight"); plt.close()
            log(">>> Shortlist priority bar written.")
except Exception as e:
    log(">>> Plotting FAILED:", e); log(traceback.format_exc())

# theme tallies on the shared-gene enrichment
def theme_counts_from_df(df):
    out = {th:0 for th in THEMES}
    if df is None or not len(df) or "Term" not in df.columns: return out
    for _,row in df.iterrows():
        t = str(row["Term"]).lower()
        for th,pat in THEMES.items():
            if re.search(pat, t, re.I): out[th]+=1
    return out
RES["shared_theme_counts"] = theme_counts_from_df(sig_shared)

# ============================================================================
# 8. DETAILED SUMMARY  ->  DOWNSTREAM_DEEP_DIVE.txt
# ============================================================================
bar="="*80; dash="-"*80
S=[]; w=lambda *a: S.append(" ".join(str(x) for x in a))
now=datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

w("#"*80)
w("#  DOWNSTREAM DEEP-DIVE SUMMARY")
w("#  Aging epigenetics (GSE190102 "+MARK+")  x  NMN-rescue (GSE85718)")
w("#  (built ONLY from 088eva.py master outputs)")
w("#"*80)
w(f"Generated     : {now}")
w(f"Master source : {MASTER_OUT}")
w(f"Output dir    : {OUT}")
w("")

# ---- KEY FINDINGS ----
w(bar); w("KEY FINDINGS  (auto-synthesized — important / significant only)"); w(bar)
w(f"[1] Shared genes carried into downstream analysis: {RES['n_shared']}")
if len(shortlist):
    top5 = shortlist.head(5)
    names = ", ".join(f"{r['gene']}(p={r['priority']:.1f})" for _,r in top5.iterrows())
    w(f"[2] Top validation candidates: {names}")
if len(sig_shared):
    w(f"[3] Shared-gene enrichment: {len(sig_shared)} significant terms (adjP<{ADJP_CUT}).")
    for _,r in sig_shared.sort_values("Adjusted P-value").head(8).iterrows():
        w(f"      - [{str(r['Library']).split('_')[0]}] {str(r['Term'])[:62]} "
          f"(adjP={fmt_p(r['Adjusted P-value'])})")
else:
    w("[3] Shared-gene enrichment: no significant terms (set may be small).")
hot = sorted([(k,v) for k,v in RES["shared_theme_counts"].items() if v>0],
             key=lambda x:-x[1])
if hot:
    w("[4] Dominant themes among shared-gene terms: " +
      "; ".join(f"{k}({v})" for k,v in hot))
w(f"[5] Mitochondrial core overlap in shared set: {len(shared_mito)} "
  f"(hypergeom p={fmt_p(RES['mito_p'])})"
  + ("  -> " + ", ".join(shared_mito) if shared_mito else ""))
if RES["string_edges"]:
    w(f"[6] STRING network on shared genes: {RES['string_nodes']} nodes / "
      f"{RES['string_edges']} edges; top hubs: "
      + ", ".join(hubs.head(8).gene.tolist()))
if split_enr:
    cc = len(split_enr.get("concordant", pd.DataFrame()))
    dd = len(split_enr.get("discordant", pd.DataFrame()))
    w(f"[7] Concordant vs discordant enrichment: {cc} vs {dd} significant terms.")
w("")
w(dash); w("Full detail follows."); w(dash); w("")

# ---- 1. Inputs ----
w(bar); w("1. INPUT TABLES (from master pipeline)"); w(bar)
for nm,df in [("shared_genes_direction", shared_dir),
              ("direction_concordance", concord),
              ("shared_significant_terms", shared_terms),
              ("NMN_robust_detail", nmn_robust),
              ("aging_ABC_targets", aging_abc)]:
    w(f"   {nm:30s}: {len(df)} rows"
      + (f"  [cols: {', '.join(df.columns)}]" if len(df) else "  (missing/empty)"))
w("")

# ---- 2. Validation shortlist ----
w(bar); w("2. PRIORITIZED VALIDATION SHORTLIST"); w(bar)
w("Priority = 3*concordant + min(n_tissues,3) + 2*ABC_rank + 2*|interaction|_rank + mito")
w("")
if len(shortlist):
    cols = [c for c in ["gene","priority","verdict","n_tissues",
                        "mean_coef_interaction","mean_coef_age_Control",
                        "ABC_score","lfc_24m_vs_3m","is_mito"] if c in shortlist.columns]
    head = "  ".join(c[:14].ljust(14) for c in cols)
    w("   "+head); w("   "+"-"*len(head))
    for _,r in shortlist.head(30).iterrows():
        cells=[]
        for c in cols:
            v=r[c]
            if isinstance(v,(int,np.integer)): s=str(int(v))
            elif isinstance(v,(float,np.floating)):
                s="NA" if pd.isna(v) else (f"{v:.2e}" if ("score" in c or c=="ABC_score")
                                           else f"{v:+.3f}" if "coef" in c or "lfc" in c
                                           else f"{v:.2f}")
            elif isinstance(v,(bool,np.bool_)): s="yes" if v else "no"
            else: s=str(v)
            cells.append(s[:14].ljust(14))
        w("   "+"  ".join(cells))
else:
    w("   (no shortlist — shared_genes_direction table absent)")
w("")

# ---- 3. Shared-gene enrichment ----
w(bar); w("3. ENRICHMENT ON SHARED GENES ONLY"); w(bar)
if len(sig_shared):
    w(f"Significant terms (adjP<{ADJP_CUT}): {len(sig_shared)}")
    for lib in ENRICHR_LIBS:
        sub = sig_shared[sig_shared["Library"]==lib].sort_values("Adjusted P-value").head(10)
        if not len(sub): continue
        w(f"\n--- {lib} ---")
        for _,r in sub.iterrows():
            w(f"   • {str(r['Term'])[:66]}  adjP={fmt_p(r['Adjusted P-value'])}")
            if "Genes" in r and pd.notna(r["Genes"]):
                gl=str(r["Genes"]).replace(";",", ")
                w(textwrap.fill(gl, width=76, initial_indent="        ",
                                subsequent_indent="        "))
else:
    w("   (no significant terms / Enrichr unavailable)")
w("")
w("Functional theme counts among significant shared-gene terms:")
for th in THEMES:
    w(f"   {th:24s} {RES['shared_theme_counts'].get(th,0)}")
w("")

# ---- 4. Concordant vs discordant ----
w(bar); w("4. CONCORDANT vs DISCORDANT ENRICHMENT"); w(bar)
if split_enr:
    for verdict in ["concordant","discordant"]:
        s = split_enr.get(verdict, pd.DataFrame())
        w(f"\n[{verdict.upper()}] significant terms: {len(s)}")
        if len(s):
            for _,r in s.sort_values("Adjusted P-value").head(10).iterrows():
                w(f"   • [{str(r['Library']).split('_')[0]}] {str(r['Term'])[:60]} "
                  f"(adjP={fmt_p(r['Adjusted P-value'])})")
else:
    w("   (concordance split not available)")
w("")

# ---- 5. Mito deep dive ----
w(bar); w("5. MITOCHONDRIAL DEEP DIVE"); w(bar)
w(f"Shared mito-core genes : {len(shared_mito)}")
w(f"Hypergeometric p       : {fmt_p(RES['mito_p'])} (mito vs shared set)")
if len(mito_df):
    w("\nShared mitochondrial genes (with direction / NMN coef):")
    cols=[c for c in mito_df.columns]
    for _,r in mito_df.iterrows():
        bits=[f"{c}={r[c]}" for c in cols if c!="gene"]
        w(f"   {r['gene']:14s} " + "  ".join(bits))
else:
    w("   (no shared mitochondrial genes)")
w("")

# ---- 6. STRING hubs ----
w(bar); w("6. STRING PPI HUBS (shared genes)"); w(bar)
if len(hubs):
    w(f"Network: {RES['string_nodes']} nodes, {RES['string_edges']} edges "
      f"(conf>={STRING_CONF}, taxon {SPECIES_ID})")
    w("\nTop hub genes by interaction degree:")
    for _,r in hubs.head(25).iterrows():
        w(f"   {r['gene']:16s} degree={int(r['degree']):3d}"
          + ("  [mito]" if r.get("is_mito") else ""))
else:
    w("   (no STRING hubs / query failed / too few genes)")
w("")

# ---- 7. Shared significant terms carried from master ----
w(bar); w("7. SHARED SIGNIFICANT TERMS (from master comparison)"); w(bar)
if len(shared_terms):
    for _,r in shared_terms.head(60).iterrows():
        lib = r.get("Library",""); term = r.get("Term","")
        w(f"   [{lib}] {term}")
else:
    w("   (master pipeline found no shared significant terms, or table absent)")
w("")

# ---- 8. Interpretation ----
w(bar); w("8. INTERPRETATION NOTES"); w(bar)
w(textwrap.fill(
  "This downstream stage adds NO new raw data; every metric is derived from the "
  "master pipeline's comparison tables. The shared-gene set is small but high-"
  "confidence (it required both an epigenetic aging signal in neurons AND a "
  "sign-reversal NMN rescue in metabolic tissues), so enrichment on it is a "
  "cleaner test of convergent biology than the full per-study sets.", width=80))
w("")
w(textwrap.fill(
  f"For the repressive mark {MARK}: a GAIN with age implies increased Polycomb "
  "repression; a coherent NMN rescue would RAISE old-age expression of that gene "
  "(positive interaction sign). 'Concordant' shortlist genes satisfy this logic "
  "and are the strongest candidates for qPCR/Western validation. With small "
  "replicate numbers upstream, treat all results as hypothesis-generating.", width=80))
w("")
w(bar); w("9. OUTPUT FILE MANIFEST"); w(bar)
for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
    if os.path.isfile(f):
        sz=os.path.getsize(f)
        szs=(f"{sz/1e6:.2f} MB" if sz>1e6 else f"{sz/1e3:.1f} KB" if sz>1e3 else f"{sz} B")
        w(f"   {f:64s} {szs:>10s}")
w("")
w(bar); w("10. RUN LOG"); w(bar)
S.extend("   "+l for l in RUN_LOG)
w(""); w("#"*80); w("#  END OF DOWNSTREAM DEEP-DIVE"); w("#"*80)

SUMMARY_PATH = f"{OUT}/DOWNSTREAM_DEEP_DIVE.txt"
with open(SUMMARY_PATH,"w") as fh: fh.write("\n".join(S)+"\n")

print("\n>>> Summary written to:", SUMMARY_PATH)
print(">>> Preview (first 70 lines):\n")
print("\n".join(S[:70]))
print("\n"+"="*60)
print("DONE — downstream outputs under:", OUT)
subprocess.run(f"find {OUT} -type f | sort", shell=True)

##############################################################################
# DOWNSTREAM DEEP-DIVE — using only 088eva.py master outputs
# Source: /content/AGING_vs_NMN_MASTER
##############################################################################
>>> loaded shared_genes_direction.tsv             :    23 rows, cols=['gene', 'aging_K27me3_dir', 'nmn_interaction_sign', 'nmn_effect']
>>> loaded direction_concordance.tsv              :    23 rows, cols=['gene', 'aging_K27me3_dir', 'nmn_sign', 'verdict']
>>> MISSING shared_significant_terms.tsv
>>> loaded NMN_robust_detail.tsv                  :    35 rows, cols=['gene', 'n_tissues', 'mean_coef_interaction', 'mean_coef_age_Control']
>>> loaded aging_ABC_targets.tsv                  : 21155 rows, cols=['peak_id', 'activity', 'Chromosome', 'Start', 'End', 'distance', 'contact', 'AxC', 'ABC_score', 'TargetGene', 'direction', 'lfc_24m_vs_3m', 'fdr_24m']
>>> Mark inferred from tables: K27me3
>>> Shared genes available for downstream: 2

CompletedProcess(args='find /content/AGING_vs_NMN_MASTER/downstream -type f | sort', returncode=0)

In [ ]:
!cp "/content/AGING_vs_NMN_MASTER" "/content/drive/MyDrive/Dr Uccello/00_Studies/088_NMN_CASES" -r

# The End